# Theme 1: Bug-Fix Adoption Trends

**RQ1:** How does AI bug-fixing *volume* change over time?  
**RQ7:** Do developers *switch agents* for bug fixing over time?

Dataset: `mabujadallah/GitHub-Agentic-PR-Dataset`  
Coverage: Dec 2024 – Feb 2026 (15 months)  
Scope: closed bug-fix PRs only (Copilot, Cursor, Claude Code, Devin + Human baseline)

In [ ]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

In [ ]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, chi_square, mann_whitney, sig_label,
    set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME1_DIR, THEME2_DIR, THEME3_DIR,
)
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [ ]:
# ── Load data ────────────────────────────────────────────────────────
df         = load_fix_prs()
agents_df  = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df   = df[~df['is_agent']].copy()
print('Agent fix PRs:', f"{len(agents_df):,}")
print('Human fix PRs:', f"{len(human_df):,}")

## RQ1: How does AI bug-fixing volume change over time?

In [ ]:
# Monthly volume: agent (all 4) vs human
monthly = df.groupby(['month', 'is_agent']).size().unstack(fill_value=0)
monthly.columns = ['Human', 'Agent']
monthly.index   = monthly.index.astype(str)
monthly['Total']    = monthly['Agent'] + monthly['Human']
monthly['Agent_%']  = (monthly['Agent'] / monthly['Total'] * 100).round(1)
print('Monthly volume (Agent vs Human):')
print(monthly.to_string())

In [ ]:
# Figure: monthly volume — agent vs human (line chart)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly.index, monthly['Agent'], 'o-', color='#1f77b4', label='Agent (all)')
ax.plot(monthly.index, monthly['Human'], 's--', color='#7f7f7f', label='Human', alpha=0.7)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff (Jul-25)')
ax.set_xlabel('Month')
ax.set_ylabel('Bug-Fix PRs')
ax.set_title('RQ1: Monthly Bug-Fix PR Volume — Agent vs Human')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq1_volume_agent_vs_human', THEME1_DIR)

In [ ]:
# Monthly volume per agent
per_agent = agents_df.groupby(['month', 'agent']).size().unstack(fill_value=0)
per_agent.index = per_agent.index.astype(str)
cols = [a for a in AGENTS if a in per_agent.columns]
per_agent = per_agent[cols]
print('Monthly volume by agent:')
print(per_agent.to_string())

In [ ]:
# Figure: per-agent stacked bar (monthly volume)
fig, ax = plt.subplots(figsize=(13, 5))
colors = [AGENT_COLORS[a] for a in per_agent.columns]
per_agent.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.8)
ax.axvline(
    per_agent.index.tolist().index('2025-07') + 0.5,
    color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff'
)
ax.set_xlabel('Month')
ax.set_ylabel('Bug-Fix PRs')
ax.set_title('RQ1: Monthly Bug-Fix Volume by Agent')
ax.legend(loc='upper left')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq1_volume_by_agent', THEME1_DIR)

## RQ7: Do developers switch agents for bug fixing over time?

Measured as each agent's **share of total agent bug-fix PRs** per month.

In [ ]:
# Agent market share: % of agent bug-fix PRs from each tool per month
share = per_agent.div(per_agent.sum(axis=1), axis=0) * 100
print('Agent market share (%) per month:')
print(share.round(1).to_string())

In [ ]:
# Figure: market share line chart
fig, ax = plt.subplots(figsize=(13, 5))
for agent in share.columns:
    ax.plot(share.index, share[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=2)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Share of Agent Bug-Fix PRs (%)')
ax.set_title('RQ7: Agent Market Share in Bug Fixing Over Time')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq7_agent_market_share', THEME1_DIR)

In [ ]:
# Figure: stacked 100% bar — proportional agent usage
fig, ax = plt.subplots(figsize=(13, 5))
colors = [AGENT_COLORS[a] for a in share.columns]
share.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Share (%)')
ax.set_ylim(0, 100)
ax.set_title('RQ7: Proportional Agent Usage for Bug Fixing (Monthly)')
ax.legend(loc='upper right')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq7_agent_share_stacked', THEME1_DIR)